# Revisión de formatos de fecha en la carpeta data

Este notebook recorre todos los archivos CSV dentro de `data` y detecta el formato de la columna de fecha para cada archivo.

Se clasifican formatos como:
- `yyyy-mm-dd`
- `dd/mm/yyyy`
- `mm/dd/yyyy`
- `dd-mm-yyyy`
- `mm-dd-yyyy`
- `yyyymm`
- `otro`

In [1]:
import csv
import re
from pathlib import Path

import pandas as pd

CANDIDATOS_RAIZ = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next(
    (ruta for ruta in CANDIDATOS_RAIZ
     if (ruta / 'data').is_dir() and (ruta / 'subte_fpca_work').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontró la raíz del proyecto (data y subte_fpca_work).')
DATA_DIR = PROJECT_ROOT / 'data'
MAX_ROWS_PER_FILE = 50
OUT_PATH = PROJECT_ROOT / 'subte_fpca_work' / 'formato_fechas_por_archivo.csv'


def split_line_to_fields(line):
    text = line.strip()
    if not text:
        return []

    for sep in [';', ',', '\t', '|']:
        try:
            row = next(csv.reader([text], delimiter=sep, quotechar='"'))
        except Exception:
            continue

        values = [c.strip() for c in row if c.strip()]
        if len(values) > 1:
            return row

        if len(values) == 1 and sep in values[0]:
            inner = values[0].strip('"').strip("'")
            try:
                inner_row = next(csv.reader([inner], delimiter=sep, quotechar='"'))
            except Exception:
                inner_row = []
            inner_values = [c.strip() for c in inner_row if c.strip()]
            if len(inner_values) > 1:
                return inner_row

    return [text]


def read_csv_sample(path, nrows=MAX_ROWS_PER_FILE):
    encodings = ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']
    last_error = None

    for enc in encodings:
        try:
            text = path.read_text(encoding=enc, errors='replace')
            rows = []

            for line in text.splitlines():
                line_clean = line.lstrip('\ufeff').strip()
                if not line_clean or line_clean.lower().startswith('sep='):
                    continue

                fields = split_line_to_fields(line_clean)
                rows.append(fields)
                if len(rows) >= nrows + 1:
                    break

            if not rows:
                continue

            header = [c.strip(' \ufeff"\'') for c in rows[0]]
            data_rows = rows[1:]
            if not data_rows:
                return pd.DataFrame(columns=header)

            df = pd.DataFrame(data_rows, columns=header)
            return df

        except Exception as e:
            last_error = e

    raise ValueError(f'No se pudo leer {path}: {last_error}')


def detect_date_format(values):
    valores = []
    for v in values:
        s = str(v).strip().lstrip('\ufeff')
        if s and s.lower() not in {'nan', 'none', 'na'}:
            valores.append(s)

    if not valores:
        return 'sin fecha'

    conteo = {
        'yyyy-mm-dd': 0,
        'dd-mm-yyyy': 0,
        'mm-dd-yyyy': 0,
        'dd/mm/yyyy': 0,
        'mm/dd/yyyy': 0,
        'yyyymm': 0,
        'otro': 0,
    }

    for v in valores[:50]:
        if re.fullmatch(r'\d{4}-\d{2}-\d{2}', v):
            conteo['yyyy-mm-dd'] += 1
        elif re.fullmatch(r'\d{6}', v):
            conteo['yyyymm'] += 1
        elif re.fullmatch(r'\d{2}-\d{2}-\d{4}', v):
            a, b, _ = map(int, v.split('-'))
            if a > 12 and b <= 12:
                conteo['dd-mm-yyyy'] += 1
            elif b > 12 and a <= 12:
                conteo['mm-dd-yyyy'] += 1
            else:
                conteo['dd-mm-yyyy'] += 0.5
                conteo['mm-dd-yyyy'] += 0.5
        elif re.fullmatch(r'\d{1,2}/\d{1,2}/\d{4}', v):
            a, b, _ = map(int, v.split('/'))
            if a > 12 and b <= 12:
                conteo['dd/mm/yyyy'] += 1
            elif b > 12 and a <= 12:
                conteo['mm/dd/yyyy'] += 1
            else:
                conteo['dd/mm/yyyy'] += 0.5
                conteo['mm/dd/yyyy'] += 0.5
        else:
            conteo['otro'] += 1

    return max(conteo, key=conteo.get)


def find_date_column(df):
    for col in df.columns:
        nombre = str(col).strip().lstrip('\ufeff').lower().replace(' ', '_')
        if 'fecha' in nombre or 'date' in nombre:
            return col

    for col in df.columns:
        sample = df[col].dropna().astype(str).head(20)
        if sample.empty:
            continue
        var = sample.tolist()
        match_count = sum(
            bool(re.search(r'\d{4}-\d{2}-\d{2}|\d{2}-\d{2}-\d{4}|\d{1,2}/\d{1,2}/\d{4}|\d{6}', str(x).strip()))
            for x in var
        )
        if match_count / len(var) >= 0.5:
            return col

    return None


resultados = []
for path in sorted(DATA_DIR.rglob('*')):
    if path.suffix.lower() != '.csv':
        continue

    try:
        df = read_csv_sample(path, nrows=MAX_ROWS_PER_FILE)
    except Exception as e:
        resultados.append({
            'archivo': str(path),
            'columna_fecha': None,
            'tipo': 'error_al_leer',
            'detalle': str(e),
        })
        continue

    col_fecha = find_date_column(df)
    if col_fecha is None:
        tipo = 'sin columna de fecha'
        muestra = None
    else:
        muestra = df[col_fecha].dropna().astype(str).head(10).tolist()
        tipo = detect_date_format(df[col_fecha])

    resultados.append({
        'archivo': str(path),
        'columna_fecha': col_fecha,
        'tipo': tipo,
        'muestra': muestra,
    })

resumen = pd.DataFrame(resultados)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
resumen[['archivo', 'columna_fecha', 'tipo']].to_csv(OUT_PATH, index=False)
print(f'Formato de fechas guardado en: {OUT_PATH.resolve()}')
resumen

Formato de fechas guardado en: /Users/nadinasoler/Documents/documentos/Nari/proyecto_subtes/subte_fpca_work/formato_fechas_por_archivo.csv


,archivo,columna_fecha,tipo,muestra
0,data/molinetes-2014.csv,FECHA,yyyy-mm-dd,"[2014-04-09, 2014-04-09, 2014-04-09, 2014-04-0..."
1,data/molinetes-2015.csv,fecha,yyyy-mm-dd,"[2015-01-01, 2015-01-01, 2015-01-01, 2015-01-0..."
2,data/molinetes-2016.csv,FECHA,dd/mm/yyyy,"[02/01/2016, 02/01/2016, 05/01/2016, 06/01/201..."
3,data/molinetes-2017.csv,FECHA,dd/mm/yyyy,"[01/01/2017, 01/01/2017, 01/01/2017, 01/01/201..."
4,data/molinetes-2018.csv,fecha,yyyy-mm-dd,"[2018-01-01, 2018-01-01, 2018-01-01, 2018-01-0..."
...,...,...,...,...
117,data/molinetes-2026/202605_PAX15min-ABC-INCLUY...,FECHA,dd/mm/yyyy,"[1/5/2026, 1/5/2026, 1/5/2026, 1/5/2026, 1/5/2..."
118,data/molinetes-2026/202605_PAX15min-DEH-INCLUY...,FECHA,dd/mm/yyyy,"[1/5/2026, 1/5/2026, 1/5/2026, 1/5/2026, 1/5/2..."
119,data/molinetes-2026/202606_PAX15min-ABC-INCLUY...,FECHA,dd/mm/yyyy,"[1/6/2026, 1/6/2026, 1/6/2026, 1/6/2026, 1/6/2..."
120,data/molinetes-2026/202606_PAX15min-DEH-INCLUY...,FECHA,dd/mm/yyyy,"[1/6/2026, 1/6/2026, 1/6/2026, 1/6/2026, 1/6/2..."
